# Session 1 — How do ENSO and the Indian Ocean Dipole interact?

**Workshop: ENSO, the IOD and floods in Bangladesh** · Session 1 of 2 · 60 minutes

In this session you will build the two climate indices yourself from sea surface temperature (SST) data, measure how strongly they are linked, look at what the ocean looks like when they occur together or apart, and let a machine-learning algorithm find "ocean states" without being told what ENSO or the IOD are.

**How to use this notebook:** run each cell in order with **Shift + Enter**. You do not need to write code. After each figure, read the *What to look for* box and talk about the questions with your neighbour. Cells marked **🔧 Try this** have one or two settings you can change and re-run.

| Time | Part |
|---|---|
| 0–10 min | Concepts (slides) and setup |
| 10–25 min | Part 1 — Build the Niño 3.4 and DMI indices |
| 25–40 min | Part 2 — How strongly are they linked, and when? |
| 40–52 min | Part 3 — Let the data find the ocean regimes (k-means) |
| 52–60 min | Part 4 — Classify every year (input for Session 2) |

**Data used:** NOAA ERSSTv5 monthly sea surface temperature (1979–present, 2° grid), cut to the tropical Indo-Pacific.

## Setup — load the workshop data

This cell finds the workshop data. In Colab it first copies the workshop repository from GitHub, which takes up to a minute the first time; after that it is instant.

In [ ]:
%pip install scikit-learn scipy shap

In [ ]:
import os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Find the workshop data. In Colab we copy (clone) the workshop repository first;
# when running from a downloaded copy of the repository, its data folder is used directly.
REPO_URL = "https://github.com/pazolka/ai-climate-workshop-dhaka.git"
DATA_DIR = None
for candidate in ["../data", "data", "ai-climate-workshop-dhaka/data"]:
    if os.path.exists(os.path.join(candidate, "gridded", "sst_indopacific.npz")):
        DATA_DIR = candidate
        break
if DATA_DIR is None:
    print("Copying the workshop repository (about 100 MB, takes up to a minute)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    DATA_DIR = "ai-climate-workshop-dhaka/data"

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty land cells give harmless warnings
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Workshop data found in:", DATA_DIR)

In [ ]:
# Helper functions (you don't need to read these — just run the cell)

def season_months(season):
    """'JJAS' -> [6, 7, 8, 9];  'DJF' -> [12, 1, 2]"""
    s = "JFMAMJJASOND" * 2
    i = s.find(season)
    if i < 0:
        raise ValueError(f"Unknown season {season}")
    return [(i + k) % 12 + 1 for k in range(len(season))]


def seasonal_mean(series, season):
    """Mean of a monthly pandas Series over a season, one value per year.
    For seasons crossing the new year (e.g. DJF) the year of January is used."""
    months = season_months(season)
    df = pd.DataFrame({"v": series.values, "m": series.index.month, "y": series.index.year})
    if months[-1] < months[0]:
        df.loc[df.m >= months[0], "y"] += 1
    sel = df[df.m.isin(months)]
    g = sel.groupby("y")["v"]
    return g.mean()[g.count() == len(months)]


def seasonal_maps(field, times, season):
    """Seasonal mean of a (time, lat, lon) array -> years, (year, lat, lon) array."""
    months = season_months(season)
    yrs = times.year.to_numpy().copy()
    if months[-1] < months[0]:
        yrs[times.month >= months[0]] += 1
    keep = np.isin(times.month, months)
    years, maps = [], []
    for y in np.unique(yrs[keep]):
        idx = keep & (yrs == y)
        if idx.sum() == len(months):
            years.append(y)
            maps.append(np.nanmean(field[idx], axis=0))
    return np.array(years), np.array(maps)


def box_mean(field, lat, lon, lat_s, lat_n, lon_w, lon_e):
    """Area-weighted average of a (time, lat, lon) array over a lat/lon box."""
    la = (lat >= lat_s) & (lat <= lat_n)
    lo = (lon >= lon_w) & (lon <= lon_e)
    sub = field[:, la][:, :, lo]
    w = np.cos(np.deg2rad(lat[la]))[:, None] * np.ones(lo.sum())[None, :]
    valid = ~np.isnan(sub)
    return np.nansum(sub * w, axis=(1, 2)) / np.sum(w * valid, axis=(1, 2))


def anomalies(field, times, base=(1991, 2020), detrend=True):
    """Remove the monthly climatology (and optionally a linear trend) at every grid point."""
    anom = np.full_like(field, np.nan)
    in_base = (times.year >= base[0]) & (times.year <= base[1])
    for m in range(1, 13):
        sel = times.month == m
        anom[sel] = field[sel] - np.nanmean(field[sel & in_base], axis=0)
    if detrend:
        t = np.arange(len(times), dtype=float)
        flat = anom.reshape(len(t), -1)
        ok = ~np.isnan(flat).any(axis=0)
        coef = np.polyfit(t, flat[:, ok], 1)
        flat[:, ok] -= np.outer(t, coef[0]) + coef[1]
        anom = flat.reshape(anom.shape)
    return anom


NINO34_BOX = dict(lat_s=-5, lat_n=5, lon_w=190, lon_e=240)
DMI_WEST = dict(lat_s=-10, lat_n=10, lon_w=50, lon_e=70)
DMI_EAST = dict(lat_s=-10, lat_n=0, lon_w=90, lon_e=110)


def plot_map(ax, field, lat, lon, title="", vmax=1.5, cmap="RdBu_r", boxes=True):
    ax.set_facecolor("0.85")  # land shows as grey
    pm = ax.pcolormesh(lon, lat, field, cmap=cmap, vmin=-vmax, vmax=vmax, shading="auto")
    if boxes:
        for b, c in [(NINO34_BOX, "k"), (DMI_WEST, "purple"), (DMI_EAST, "purple")]:
            ax.add_patch(Rectangle((b["lon_w"], b["lat_s"]), b["lon_e"] - b["lon_w"],
                                   b["lat_n"] - b["lat_s"], fill=False, ec=c, lw=1.2))
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=10, loc="left")
    ax.grid(False)
    ax.set_xticks(range(40, 300, 40))
    ax.set_xticklabels([f"{x}°E" if x <= 180 else f"{360 - x}°W" for x in range(40, 300, 40)], fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    return pm

print("Helpers ready.")

## Part 1 — Build the two indices (≈15 min)

**ENSO** is measured with the **Niño 3.4 index**: the average SST anomaly in the central-eastern equatorial Pacific (5°S–5°N, 170°W–120°W). Positive = El Niño, negative = La Niña.

**The IOD** is measured with the **Dipole Mode Index (DMI)**: the SST anomaly in the western tropical Indian Ocean (50–70°E, 10°S–10°N) *minus* the anomaly in the south-eastern box (90–110°E, 10°S–0°). Positive IOD = warm west, cool east.

First we turn raw SST into **anomalies**: the difference from the normal value for that calendar month (1991–2020 average). We also remove the slow warming trend, so that the indices describe year-to-year swings rather than climate change.

In [ ]:
d = np.load(f"{DATA_DIR}/gridded/sst_indopacific.npz")
sst, lat, lon = d["sst"], d["lat"], d["lon"]
times = pd.to_datetime(pd.Series(d["time"]).astype(str) + "-01")
times = pd.DatetimeIndex(times)
print(f"SST: {sst.shape[0]} months ({times[0]:%b %Y} – {times[-1]:%b %Y}), "
      f"{len(lat)} latitudes × {len(lon)} longitudes")

sst_anom = anomalies(sst, times, detrend=True)

nino34 = pd.Series(box_mean(sst_anom, lat, lon, **NINO34_BOX), index=times, name="Niño 3.4")
dmi = pd.Series(box_mean(sst_anom, lat, lon, **DMI_WEST) - box_mean(sst_anom, lat, lon, **DMI_EAST),
                index=times, name="DMI")

# Where are the boxes? A map of one month's anomaly (October 1997: El Niño + positive IOD)
fig, ax = plt.subplots(figsize=(12, 3))
k = np.where((times.year == 1997) & (times.month == 10))[0]
k = k[0] if len(k) else len(times) - 1
pm = plot_map(ax, sst_anom[k], lat, lon, f"SST anomaly, {times[k]:%B %Y} (°C). Black box: Niño 3.4. Purple boxes: IOD west and east", vmax=3)
plt.colorbar(pm, ax=ax, shrink=0.8, label="°C")
plt.show()

In [ ]:
# Plot both indices (3-month running means, as the official indices use)
n3 = nino34.rolling(3, center=True).mean()
d3 = dmi.rolling(3, center=True).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
for ax, s, name, thr in [(axes[0], n3, "Niño 3.4 (°C)", 0.5), (axes[1], d3, "DMI (°C)", 0.4)]:
    ax.fill_between(s.index, s, 0, where=s > 0, color="tab:red", alpha=0.6, lw=0)
    ax.fill_between(s.index, s, 0, where=s < 0, color="tab:blue", alpha=0.6, lw=0)
    for t in (thr, -thr):
        ax.axhline(t, color="k", lw=0.6, ls="--")
    ax.set_ylabel(name)
axes[0].set_title("ENSO (top) and the Indian Ocean Dipole (bottom). Dashed lines: usual event thresholds", loc="left")
for y in [1997, 2006, 2015, 2019, 2023]:
    for ax in axes:
        ax.axvline(pd.Timestamp(f"{y}-10-15"), color="0.4", lw=0.8)
    axes[0].text(pd.Timestamp(f"{y}-10-15"), axes[0].get_ylim()[1] * 0.9, str(y), fontsize=8, ha="center",
                 bbox=dict(fc="white", ec="none", pad=1))
plt.tight_layout(); plt.show()

In [ ]:
# Sanity check: do our home-made indices agree with the official NOAA ones?
checks, official = [], {}
if os.path.exists(f"{DATA_DIR}/oni_official.csv"):
    oni = pd.read_csv(f"{DATA_DIR}/oni_official.csv")
    centre = {s: i + 1 for i, s in enumerate(["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA",
                                               "JAS", "ASO", "SON", "OND", "NDJ"])}
    oni.index = pd.to_datetime(dict(year=oni.year, month=oni.season.map(centre), day=1))
    official["oni"] = oni["anom"]
    both = pd.concat([n3, oni["anom"]], axis=1, sort=True).dropna()
    checks.append(f"Our Niño 3.4 vs official ONI:   r = {both.corr().iloc[0, 1]:.2f}")
if os.path.exists(f"{DATA_DIR}/dmi_official.csv"):
    dmi_off = pd.read_csv(f"{DATA_DIR}/dmi_official.csv")
    dmi_off.index = pd.to_datetime(dmi_off["month"] + "-01")
    official["dmi"] = dmi_off["dmi"]
    both = pd.concat([dmi, dmi_off["dmi"]], axis=1, sort=True).dropna()
    son = both[both.index.month.isin([9, 10, 11])]
    checks.append(f"Our DMI vs official NOAA DMI:   r = {both.corr().iloc[0, 1]:.2f} (all months), "
                  f"r = {son.corr().iloc[0, 1]:.2f} (September–November only)")
print("\n".join(checks) if checks else "Official index files not included — skipping the check.")

# Plot ours and NOAA's together (3-month running means, so the lines are easier to compare)
if official:
    fig, axes = plt.subplots(len(official), 1, figsize=(12, 2.8 * len(official)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    pairs = []
    if "oni" in official:
        pairs.append((n3, official["oni"], "Niño 3.4 / ONI (°C)", "NOAA ONI"))
    if "dmi" in official:
        pairs.append((d3, official["dmi"].rolling(3, center=True).mean(), "DMI (°C)", "NOAA DMI (HadISST)"))
    for ax, (ours, theirs, label, name) in zip(axes, pairs):
        theirs = theirs[theirs.index >= ours.index[0]]
        ax.plot(theirs.index, theirs, color="0.55", lw=2.2, label=name)
        ax.plot(ours.index, ours, color="tab:red" if "Niño" in label else "tab:purple", lw=1.1, label="ours (ERSSTv5)")
        ax.axhline(0, color="k", lw=0.5)
        ax.set_ylabel(label)
        ax.legend(loc="upper left", fontsize=8, ncol=2)
    axes[0].set_title("Our indices (thin lines) and NOAA's official ones (thick grey)", loc="left")
    plt.tight_layout(); plt.show()

In [ ]:
# Zoom in on September–November, when IOD events happen: one value per year (the SON average)
if "dmi" in official:
    ours_son = seasonal_mean(dmi, "SON")
    noaa_son = seasonal_mean(official["dmi"], "SON")
    son_pair = pd.concat([ours_son.rename("ours"), noaa_son.rename("noaa")], axis=1, sort=True).dropna()
    r_son = son_pair.corr().iloc[0, 1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2.2, 1]})
    ax = axes[0]
    ax.plot(son_pair.index, son_pair["noaa"], "o-", color="0.55", lw=2.2, ms=5, label="NOAA DMI (HadISST)")
    ax.plot(son_pair.index, son_pair["ours"], "o-", color="tab:purple", lw=1.1, ms=3.5, label="ours (ERSSTv5)")
    ax.axhline(0, color="k", lw=0.5)
    for t in (0.4, -0.4):
        ax.axhline(t, color="k", lw=0.6, ls="--")
    ax.set_ylabel("DMI, Sep–Nov average (°C)")
    ax.set_title("September–November DMI, one value per year (dashed: usual event threshold)", loc="left")
    ax.legend(loc="upper left", fontsize=8)

    ax = axes[1]
    ax.scatter(son_pair["noaa"], son_pair["ours"], s=35, color="tab:purple", edgecolor="k", lw=0.5)
    for y in [1997, 2006, 2016, 2019, 2023]:
        if y in son_pair.index:
            ax.annotate(str(y), (son_pair.loc[y, "noaa"], son_pair.loc[y, "ours"]), fontsize=8,
                        xytext=(3, 3), textcoords="offset points")
    lim = [son_pair.min().min() - 0.15, son_pair.max().max() + 0.15]
    ax.plot(lim, lim, "k--", lw=0.8, label="perfect match")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.set_xlabel("NOAA DMI (°C)"); ax.set_ylabel("our DMI (°C)")
    ax.set_title(f"Sep–Nov: r = {r_son:.2f} ({len(son_pair)} years)", loc="left")
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("Official DMI file not included — skipping this plot.")

> **What to look for**
> - Find **1997** and **2015**. What are both indices doing in those years? Now find **2019** and **2006**. Do the two indices always move together?
> - In which months does each index reach its biggest values? Do ENSO and the IOD peak at the same time of year?
> - **The check against NOAA:** how closely do our indices match NOAA's official ones? Is the match equally good for ENSO and for the IOD, and is it better in some seasons? In the September–November scatter plot, are the strongest IOD years above or below the dashed line? What might explain any difference? (Hint: NOAA builds its DMI from a different temperature dataset, HadISST.)
>
> **Discuss:** Do positive IOD events seem to happen more often during El Niño? Can you find exceptions?

## Part 2 — How strongly are they linked, and when? (≈15 min)

### 2a. Correlation by season

The chart below compares Niño 3.4 in one 3-month season (rows) with the DMI in another season *of the same year* (columns). Each square is a correlation across all years: red = they tend to have the same sign, blue = opposite signs.

In [ ]:
names = ["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA", "JAS", "ASO", "SON", "OND", "NDJ"]

def by_year_month(s):
    df = pd.DataFrame({"v": s.values, "y": s.index.year, "m": s.index.month})
    return df.pivot(index="y", columns="m", values="v")

N = by_year_month(n3)
D = by_year_month(d3)
lag_corr = np.full((12, 12), np.nan)
for i in range(12):
    for j in range(12):
        pair = pd.concat([N[i + 1], D[j + 1]], axis=1).dropna()
        lag_corr[i, j] = pair.corr().iloc[0, 1]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(lag_corr, cmap="RdBu_r", vmin=-0.8, vmax=0.8, origin="lower")
ax.set_xticks(range(12)); ax.set_xticklabels(names, rotation=45)
ax.set_yticks(range(12)); ax.set_yticklabels(names)
ax.set_xlabel("DMI season"); ax.set_ylabel("Niño 3.4 season")
for i in range(12):
    for j in range(12):
        ax.text(j, i, f"{lag_corr[i, j]:.1f}", ha="center", va="center", fontsize=6.5)
ax.grid(False)
ax.set_title("Correlation: Niño 3.4 vs DMI (same calendar year)", loc="left")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.show()

print(f"Same-season correlation in summer (JJA):        {lag_corr[6, 6]:.2f}")
print(f"Same-season correlation at the IOD peak (SON): {lag_corr[9, 9]:.2f}")

> **What to look for**
> - Follow the diagonal (the same season for both indices). In which season is the link weakest, and in which is it strongest?
> - Pick the DMI in autumn (the SON column). Is it more closely linked to Niño 3.4 in the spring before, or in the summer before? What does that say about how the two modes develop?
> - What is the largest correlation you can find?
>
> **Discuss:** Take the largest correlation and square it: that is the fraction of the IOD's year-to-year variance that ENSO "explains". How much of the IOD is left for other causes?

### 2b. Has the relationship been stable over time?

Here we compute the SON correlation in a sliding 21-year window. If the link were fixed, the line would be flat.

In [ ]:
WINDOW = 21   # 🔧 Try this: 15 or 25 years
SEASON_STAB = "SON"

pair = pd.concat([seasonal_mean(nino34, SEASON_STAB), seasonal_mean(dmi, SEASON_STAB)], axis=1).dropna()
pair.columns = ["nino34", "dmi"]
roll = pair["nino34"].rolling(WINDOW, center=True).corr(pair["dmi"])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(roll.index, roll, "o-", color="tab:purple")
ax.axhline(pair.corr().iloc[0, 1], color="k", ls="--", lw=0.8, label="whole period")
ax.set_ylim(-0.2, 1); ax.set_ylabel("correlation")
ax.set_title(f"{WINDOW}-year running correlation, {SEASON_STAB} Niño 3.4 vs DMI (plotted at window centre)", loc="left")
ax.legend(); plt.show()

> **What to look for:** Does the line stay roughly flat, or does it drift? With only about 47 years of data, a 21-year window holds just 21 points, so swings of ±0.2 can happen by chance. Treat any trend as a hypothesis, not a finding.

### 2c. What does the ocean look like in each combination?

We now label each year by its ENSO and IOD state in a chosen season, then average the SST anomaly maps for groups of years (**composites**).

We use a simple rule on the standardised seasonal index: above **+0.75 standard deviations** = El Niño (or positive IOD), below −0.75 = La Niña (or negative IOD), otherwise neutral. Using standard deviations, rather than a fixed °C value, treats the two indices fairly, because the IOD is naturally smaller than ENSO.

In [ ]:
SEASON = "SON"     # 🔧 Try this: "JJAS" (the monsoon season) or "JJA"
THRESHOLD = 0.75   # 🔧 Try this: 0.5 or 1.0 (in standard deviations)

def classify(x, thr, pos, neg):
    z = (x - x.mean()) / x.std()
    return pd.Series(np.where(z > thr, pos, np.where(z < -thr, neg, "Neutral")), index=x.index)

yr_n34 = seasonal_mean(nino34, SEASON)
yr_dmi = seasonal_mean(dmi, SEASON)
enso = classify(yr_n34, THRESHOLD, "El Niño", "La Niña")
iod = classify(yr_dmi, THRESHOLD, "IOD+", "IOD-")
enso.name, iod.name = "ENSO", "IOD"

table = pd.crosstab(enso, iod).reindex(index=["El Niño", "Neutral", "La Niña"],
                                       columns=["IOD+", "Neutral", "IOD-"], fill_value=0)
print(f"Number of years in each combination ({SEASON}, threshold ±{THRESHOLD} SD):\n")
print(table, "\n")
for e in ["El Niño", "La Niña"]:
    for i in ["IOD+", "IOD-"]:
        yrs = list(enso.index[(enso == e) & (iod == i)])
        print(f"{e:8s} + {i}: {yrs}")

In [ ]:
years_m, maps = seasonal_maps(sst_anom, times, SEASON)
maps = pd.Series(list(maps), index=years_m)

groups = {
    "El Niño + positive IOD": (enso == "El Niño") & (iod == "IOD+"),
    "El Niño, IOD not positive": (enso == "El Niño") & (iod != "IOD+"),
    "Positive IOD without El Niño": (iod == "IOD+") & (enso != "El Niño"),
    "La Niña + negative IOD": (enso == "La Niña") & (iod == "IOD-"),
}

fig, axes = plt.subplots(len(groups), 1, figsize=(12, 2.9 * len(groups)), layout="constrained")
for ax, (title, mask) in zip(axes, groups.items()):
    yrs = [y for y in mask.index[mask] if y in maps.index]
    if yrs:
        comp = np.nanmean(np.stack([maps[y] for y in yrs]), axis=0)
        pm = plot_map(ax, comp, lat, lon, f"{title}  (n = {len(yrs)}: {', '.join(map(str, yrs))})", vmax=1.5)
    else:
        ax.text(0.5, 0.5, f"{title}: no years", transform=ax.transAxes, ha="center")
        ax.set_axis_off()
fig.colorbar(pm, ax=axes, shrink=0.6, label=f"{SEASON} SST anomaly (°C)")
plt.show()

> **What to look for**
> - **El Niño + positive IOD:** describe the pattern in the Pacific and in the Indian Ocean. Is there a link across Indonesia?
> - **Positive IOD without El Niño:** does the Indian Ocean pattern appear on its own? How many years are in this group?
> - **Check the n:** how many years does each map average? Could one unusual year dominate a map?
>
> **Discuss:** Why might an El Niño "push" the Indian Ocean toward a positive IOD? (Think about the Walker circulation and winds over Indonesia.)

## Part 3 — Let the data find the regimes: k-means clustering (≈12 min)

So far *we* decided what counts as El Niño or a positive IOD. Now we ask an algorithm to group years by the **whole SST map**, without telling it anything about ENSO or the IOD.

Steps (all done in the next cell):
1. Take one SST anomaly map per year (for the chosen season).
2. Compress each map with **PCA** (principal component analysis) into a few numbers that capture the main patterns.
3. Group the years with **k-means** into `N_CLUSTERS` clusters of similar maps.

Then we check whether the clusters line up with our ENSO–IOD labels.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

N_CLUSTERS = 4   # 🔧 Try this: 3, 5 or 6
N_PCS = 5        # number of PCA patterns kept

X_maps = np.stack(maps.values)                       # (years, lat, lon)
ocean = ~np.isnan(X_maps).any(axis=0)
w = np.sqrt(np.cos(np.deg2rad(lat)))[:, None] * np.ones(len(lon))[None, :]
X = (X_maps * w)[:, ocean]                           # (years, ocean points)

pca = PCA(n_components=N_PCS)
pcs = pca.fit_transform(X)
print("Variance explained by each PCA pattern:", np.round(pca.explained_variance_ratio_ * 100, 1), "%")

km = KMeans(n_clusters=N_CLUSTERS, n_init=50, random_state=0).fit(pcs)
labels = pd.Series(km.labels_, index=maps.index)

# Order clusters from coldest to warmest Pacific, so the numbering is easy to read
order = yr_n34.reindex(labels.index).groupby(labels).mean().sort_values().index
relabel = {old: new + 1 for new, old in enumerate(order)}
cluster = labels.map(relabel).rename("cluster")

fig, axes = plt.subplots(N_CLUSTERS, 1, figsize=(12, 2.9 * N_CLUSTERS), layout="constrained")
axes = np.atleast_1d(axes)
for c, ax in zip(range(1, N_CLUSTERS + 1), axes):
    yrs = list(cluster.index[cluster == c])
    comp = np.nanmean(np.stack([maps[y] for y in yrs]), axis=0)
    mean_n, mean_d = yr_n34.reindex(yrs).mean(), yr_dmi.reindex(yrs).mean()
    pm = plot_map(ax, comp, lat, lon,
                  f"Cluster {c}  (n={len(yrs)}; mean Niño 3.4 {mean_n:+.1f}, mean DMI {mean_d:+.2f})  "
                  f"{', '.join(map(str, yrs))}", vmax=1.5)
fig.colorbar(pm, ax=axes, shrink=0.6, label=f"{SEASON} SST anomaly (°C)")
plt.show()

In [ ]:
# How do the clusters compare with our index-based labels?
combo = (enso + " / " + iod).reindex(cluster.index).rename("ENSO / IOD")
print(pd.crosstab(cluster, combo), "\n")

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(yr_n34.reindex(cluster.index), yr_dmi.reindex(cluster.index), c=cluster,
                cmap="viridis", s=60, edgecolor="k")
for y in cluster.index:
    ax.annotate(str(y)[2:], (yr_n34[y], yr_dmi[y]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel(f"Niño 3.4, {SEASON} (°C)"); ax.set_ylabel(f"DMI, {SEASON} (°C)")
ax.set_title("Each dot is a year, coloured by its k-means cluster", loc="left")
plt.colorbar(sc, ax=ax, label="cluster", ticks=range(1, N_CLUSTERS + 1))
plt.show()

> **What to look for**
> - Does one cluster look like "El Niño + positive IOD" and another like "La Niña + negative IOD"? Is there a cluster that is mostly an Indian Ocean signal?
> - In the scatter plot, are clusters separated mainly along the ENSO axis (left–right), the IOD axis (up–down), or diagonally?
> - Run the clustering again with 3, 5 or 6 clusters. Which patterns are robust, and which appear only sometimes?
>
> **Discuss:** The algorithm found these groups without knowing anything about ENSO or the IOD. What are the benefits of letting the data define regimes? What are the risks with only ~47 maps? (Think: is the algorithm finding physics, or just splitting a cloud of points?)

## Part 4 — Classify every year and save the table (≈8 min)

Session 2 uses these labels to ask whether flood years in Bangladesh line up with particular ENSO–IOD combinations. We save both the **monsoon season (JJAS)** classification, which matters for floods, and the **autumn (SON)** classification, when the IOD peaks.

In [ ]:
rows = {}
for s in ["JJAS", "SON"]:
    n_s, d_s = seasonal_mean(nino34, s), seasonal_mean(dmi, s)
    rows[f"nino34_{s}"] = n_s.round(2)
    rows[f"dmi_{s}"] = d_s.round(2)
    rows[f"enso_{s}"] = classify(n_s, 0.75, "El Niño", "La Niña")
    rows[f"iod_{s}"] = classify(d_s, 0.75, "IOD+", "IOD-")
summary = pd.DataFrame(rows)
summary[f"kmeans_cluster_{SEASON}"] = cluster
summary.index.name = "year"
summary.to_csv("enso_iod_years.csv")
print("Saved enso_iod_years.csv (folder icon on the left to download)\n")
summary.tail(15)

## Wrap-up

Before Session 2, discuss with your neighbour:

1. **Are ENSO and the IOD linked?** How strongly, and in which season? Are they the same thing?
2. **When do they grow?** Do they build up together through the monsoon season? Why would that matter for Bangladesh?
3. **Can the IOD act alone?** How often did you find a positive IOD without El Niño?
4. **Did the algorithm agree with our definitions?** Which groups did it find, and which did it miss?
5. **How sure can we be?** How many years did each group or cluster contain? Which findings would you call robust, and which are hypotheses to test with more data?

**In Session 2** we connect these ocean states to monsoon rainfall and to river flow on the Brahmaputra and Ganges, and test whether knowing *both* indices helps predict flood years.